# Notebook 01 — Tox21 Multi-task Benchmark
**Author: Himanshu Goel** | [Website](https://himanshugoel.github.io)

The **Tox21** dataset (NIH/EPA/FDA challenge, 2014) is the de-facto industry benchmark for computational toxicology. ~8 000 compounds measured across **12 nuclear-receptor and stress-response endpoints**.

| Panel | Endpoint | Biology |
|-------|----------|---------|
| Nuclear Receptor | NR-AR | Androgen receptor agonism |
| Nuclear Receptor | NR-AhR | Aryl hydrocarbon receptor |
| Nuclear Receptor | NR-ER | Estrogen receptor α agonism |
| Nuclear Receptor | NR-PPAR-gamma | Peroxisome proliferator-activated receptor |
| Stress Response | SR-ARE | Antioxidant / Nrf2 response element |
| Stress Response | SR-ATAD5 | Genotoxicity proxy (replication stress) |
| Stress Response | SR-MMP | Mitochondrial membrane potential |
| Stress Response | SR-p53 | DNA damage / p53 activation |

**Architecture**: DeepTox-style multi-task DNN (winner of the 2014 Tox21 Challenge).  
Reference: Mayr A et al. *Front Environ Sci* 2016; doi: 10.3389/fenvs.2016.00080

In [ ]:
!pip install deepchem rdkit scikit-learn pandas numpy matplotlib seaborn torch -q

In [ ]:
# -- SSL fix (macOS Python.org installer) -------------------------------------
# Python from python.org on macOS ships without system SSL certificates.
# Permanent fix: run /Applications/Python\ 3.12/Install\ Certificates.command
# Notebook workaround below disables verification for dataset download only.
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

# -- Reproducibility ----------------------------------------------------------
# Fix all random seeds before any data splitting or weight initialisation.
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
import deepchem as dc

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | PyTorch {torch.__version__} | DeepChem {dc.__version__}')

# -- Data loading --------------------------------------------------------------
# DeepChem's MolNet loader handles: download, ECFP4 featurisation (1024-bit),
# and scaffold-based splitting in one call.
#
# Scaffold split (Bemis-Murcko) is the industry standard for QSAR evaluation:
# it places structurally similar compounds in the same split, preventing
# information leakage that inflates AUC by 5-15% vs. random split.
# Ref: Wu Z et al. MoleculeNet. Chem Sci 2018; doi: 10.1039/C7SC02664A
print('Loading Tox21 via DeepChem MolNet (ECFP4, scaffold split)...')
tox21_tasks, datasets, transformers = dc.molnet.load_tox21(
    featurizer='ECFP',    # ECFP4, 1024-bit Morgan fingerprints
    splitter='scaffold',  # Bemis-Murcko scaffold split
)
train_ds, valid_ds, test_ds = datasets

print(f'Tasks ({len(tox21_tasks)}): {tox21_tasks}')
print(f'Train: {len(train_ds):,} | Valid: {len(valid_ds):,} | Test: {len(test_ds):,}')
print(f'Feature dim: {train_ds.X.shape[1]}')

## Exploratory data analysis — class balance per endpoint

Tox21 labels are **massively missing** (30–50% per endpoint): a compound may be
tested on only a subset of assays. The weight matrix `w` encodes this:
`w[i,j] = 0` means compound *i* was not tested on endpoint *j* and must be
**masked** from both loss and evaluation — never imputed as negative.

In [ ]:
# -- Class balance analysis ---------------------------------------------------
# Compute active rate only over compounds that were actually tested (w > 0).
# Active rates <5% trigger cost-sensitive training in industry practice.

y_tr, w_tr = train_ds.y, train_ds.w
stats = []
for i, task in enumerate(tox21_tasks):
    mask   = w_tr[:, i] > 0
    vals   = y_tr[:, i][mask]
    n_pos  = int(vals.sum())
    n_neg  = int((vals == 0).sum())
    n_miss = int((w_tr[:, i] == 0).sum())
    stats.append({
        'Endpoint':        task,
        'N_active':        n_pos,
        'N_inactive':      n_neg,
        'N_missing':       n_miss,
        'Total_measured':  n_pos + n_neg,
        'Active_pct':      round(n_pos / (n_pos + n_neg) * 100, 1) if (n_pos + n_neg) > 0 else 0,
    })

df_eda = pd.DataFrame(stats).sort_values('Active_pct')
print(df_eda[['Endpoint','N_active','N_inactive','N_missing','Active_pct']].to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Colour bands: red < 8%, orange < 15%, green >= 15%
palette = ['#e74c3c' if r < 8 else '#f39c12' if r < 15 else '#27ae60'
           for r in df_eda['Active_pct']]
ax1.barh(df_eda['Endpoint'], df_eda['Active_pct'], color=palette)
ax1.axvline(15, color='k', linestyle='--', lw=0.8, label='15% guideline')
ax1.set_xlabel('% Active (measured compounds only)')
ax1.set_title('Tox21 — Class balance per endpoint')
ax1.legend()

# Set locator before labels to avoid matplotlib FixedFormatter warning
x_pos = range(len(df_eda))
ax2.bar(x_pos, df_eda['Total_measured'], color='#2c3e50', alpha=0.7, label='Measured')
ax2.bar(x_pos, df_eda['N_missing'], bottom=df_eda['Total_measured'],
        color='#bdc3c7', alpha=0.5, label='Missing label')
ax2.set_xticks(list(x_pos))
ax2.set_xticklabels(df_eda['Endpoint'].tolist(), rotation=45, ha='right')
ax2.set_ylabel('Compound count')
ax2.set_title('Data availability (grey = missing label)')
ax2.legend()

plt.tight_layout()
plt.savefig('tox21_eda.png', dpi=150)
plt.show()

## DeepTox multi-task DNN

Architecture from the **2014 Tox21 Challenge winners** (Mayr et al.):
shared representation layers + per-task output heads.  
Multi-task learning provides implicit data augmentation — endpoints share
structural features, so rare positives in one task benefit from signal in related tasks.

In [ ]:
# -- DeepTox architecture -----------------------------------------------------
# Shared trunk -> per-task sigmoid heads.
# BatchNorm before ReLU follows the original DeepTox paper order.
# Dropout (0.35) is the value reported in Mayr et al. as optimal for Tox21.

class DeepToxNet(nn.Module):
    def __init__(self, in_features: int, n_tasks: int,
                 hidden: list = None, dropout: float = 0.35):
        super().__init__()
        # Avoid mutable default argument bug — create list inside __init__
        if hidden is None:
            hidden = [2048, 1024, 512, 256]

        layers = []
        dim = in_features
        for h in hidden:
            layers += [nn.Linear(dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            dim = h
        self.trunk = nn.Sequential(*layers)
        # Independent head per task: allows per-endpoint loss weighting
        self.heads = nn.ModuleList([nn.Linear(dim, 1) for _ in range(n_tasks)])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.trunk(x)
        return torch.cat([head(h) for head in self.heads], dim=1)  # (batch, n_tasks)


def masked_bce(pred: torch.Tensor, target: torch.Tensor,
               weight: torch.Tensor) -> torch.Tensor:
    """BCE over (compound, task) pairs that have a measured label (w > 0).
    Returns differentiable zero for all-masked batches to prevent NaN gradients.
    """
    mask = weight > 0
    if not mask.any():
        return (pred * 0.0).sum()   # differentiable zero
    return F.binary_cross_entropy_with_logits(pred[mask], target[mask])


# -- Tensor preparation -------------------------------------------------------
X_tr = torch.FloatTensor(train_ds.X).to(DEVICE)
y_tr = torch.FloatTensor(train_ds.y).to(DEVICE)
w_tr = torch.FloatTensor(train_ds.w).to(DEVICE)
X_va = torch.FloatTensor(valid_ds.X).to(DEVICE)

# -- Model, optimiser, scheduler ----------------------------------------------
model = DeepToxNet(in_features=X_tr.shape[1], n_tasks=len(tox21_tasks)).to(DEVICE)

optimiser = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

# ReduceLROnPlateau monitors validation AUC; more adaptive than fixed StepLR.
# patience=3: LR halves if val AUC does not improve for 3 consecutive epochs.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser, mode='max', factor=0.5, patience=3, verbose=False
)

loader = DataLoader(
    TensorDataset(X_tr, y_tr, w_tr),
    batch_size=256, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'DeepToxNet — trainable parameters: {n_params:,}')
print(f'Tasks: {len(tox21_tasks)} | Input dim: {X_tr.shape[1]}')

In [ ]:
# -- Training loop (20 epochs) ------------------------------------------------
# Best-model checkpointing: save weights whenever validation AUC improves.
# Gradient clipping (max_norm=1.0) prevents exploding gradients — standard
# practice for deep MLPs on sparse binary fingerprints.

EPOCHS = 20
history = {'epoch': [], 'train_loss': [], 'val_auc': [], 'lr': []}
best_val_auc = 0.0
best_state   = None

for epoch in range(EPOCHS):
    # -- Train ----------------------------------------------------------------
    model.train()
    epoch_loss = 0.0
    for xb, yb, wb in loader:
        optimiser.zero_grad()
        loss = masked_bce(model(xb), yb, wb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(loader)

    # -- Validate -------------------------------------------------------------
    model.eval()
    with torch.no_grad():
        val_probs = torch.sigmoid(model(X_va)).cpu().numpy()

    aucs = []
    for i in range(len(tox21_tasks)):
        mask   = valid_ds.w[:, i] > 0
        labels = valid_ds.y[:, i][mask]
        # Require >=10 samples and both classes present for a reliable AUC
        if mask.sum() >= 10 and len(np.unique(labels)) > 1:
            try:
                aucs.append(roc_auc_score(labels, val_probs[:, i][mask]))
            except Exception:
                pass

    mean_val_auc = float(np.mean(aucs)) if aucs else 0.0

    # -- Scheduler called after validation (not before) -----------------------
    scheduler.step(mean_val_auc)
    current_lr = optimiser.param_groups[0]['lr']

    if mean_val_auc > best_val_auc:
        best_val_auc = mean_val_auc
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}

    history['epoch'].append(epoch)
    history['train_loss'].append(avg_loss)
    history['val_auc'].append(mean_val_auc)
    history['lr'].append(current_lr)

    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f'Epoch {epoch:2d}/{EPOCHS-1} | Loss={avg_loss:.4f} | '
              f'Val AUC={mean_val_auc:.4f} | LR={current_lr:.2e}')

# Restore best checkpoint before test evaluation
model.load_state_dict(best_state)
print(f'\nRestored best checkpoint — Val AUC = {best_val_auc:.4f}')

# -- Training curves ----------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['epoch'], history['train_loss'], color='#e74c3c', lw=1.5)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Masked BCE loss')
ax1.set_title('Training loss')

ax2.plot(history['epoch'], history['val_auc'], color='#2980b9', lw=1.5)
ax2.axhline(best_val_auc, color='k', linestyle='--', lw=0.8,
            label=f'Best = {best_val_auc:.4f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Mean Val AUC')
ax2.set_title('Validation AUC (mean over tasks)')
ax2.legend()

plt.tight_layout()
plt.savefig('tox21_training_curves.png', dpi=150)
plt.show()

In [ ]:
# -- Test set evaluation ------------------------------------------------------
# Per-endpoint AUC on the held-out scaffold test split.
# Bug fix: `if auc` would treat auc==0.0 as None; use `is not None` instead.
# Bug fix: .dropna().mean() used to exclude endpoints with insufficient data.

X_te = torch.FloatTensor(test_ds.X).to(DEVICE)
model.eval()
with torch.no_grad():
    test_probs = torch.sigmoid(model(X_te)).cpu().numpy()

results = []
for i, task in enumerate(tox21_tasks):
    mask   = test_ds.w[:, i] > 0
    n_meas = int(mask.sum())
    if n_meas >= 5 and len(np.unique(test_ds.y[:, i][mask])) > 1:
        try:
            auc = roc_auc_score(test_ds.y[:, i][mask], test_probs[:, i][mask])
        except Exception as e:
            print(f'  [{task}] AUC error: {e}')
            auc = None
    else:
        auc = None
    results.append({
        'Endpoint':   task,
        'Test_AUC':   round(auc, 4) if auc is not None else None,
        'N_measured': n_meas,
    })

res_df = pd.DataFrame(results).sort_values('Test_AUC', ascending=False)
print(res_df.to_string(index=False))

valid_tasks = res_df['Test_AUC'].notna().sum()
mean_auc    = res_df['Test_AUC'].dropna().mean()
print(f'\nMean test AUC ({valid_tasks} tasks with sufficient data): {mean_auc:.4f}')
print(f'Published DeepTox benchmark (Mayr et al. 2016):           ~0.846')

# -- Per-endpoint AUC bar chart -----------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4))
plot_df    = res_df.dropna(subset=['Test_AUC'])
bar_colors = ['#27ae60' if v >= 0.8 else '#f39c12' if v >= 0.7 else '#e74c3c'
              for v in plot_df['Test_AUC']]
ax.barh(plot_df['Endpoint'], plot_df['Test_AUC'], color=bar_colors)
ax.axvline(0.8, color='green',  linestyle='--', lw=0.8, label='AUC = 0.80')
ax.axvline(0.7, color='orange', linestyle='--', lw=0.8, label='AUC = 0.70')
ax.set_xlim([0.5, 1.0])
ax.set_xlabel('Test ROC-AUC')
ax.set_title('DeepTox Multi-task DNN — Test AUC per Tox21 endpoint')
ax.legend()
plt.tight_layout()
plt.savefig('tox21_auc.png', dpi=150)
plt.show()

## Key takeaways

- **Scaffold split** is mandatory for realistic QSAR evaluation — random split inflates AUC by 5–15% by placing similar compounds in both train and test.
- **Missing labels must be masked** (weight matrix `w`) — never impute zeros; imputed negatives dominate the loss and degrade rare-endpoint performance.
- **Multi-task learning** improves rare endpoints (NR-AR, SR-ATAD5) by sharing representations across 12 related toxicity assays.
- **Best-model checkpointing** on validation AUC prevents final-epoch overfitting; ReduceLROnPlateau adapts the LR to plateau behaviour.
- **Gradient clipping** (norm ≤ 1.0) is standard for deep ECFP-DNN models — sparse fingerprint inputs can produce large gradient magnitudes early in training.
- **SR-MMP** (mitochondrial membrane potential) is typically the easiest endpoint (~AUC 0.87); **NR-AR** the hardest (~0.72) — consistent with literature.
- **Industry tools**: chemprop (D-MPNN), AttentiveFP, DeepTox, Tox21NN — all benchmark on this dataset.